# RSNA R100 hybrid — mounted-data smoke

Pre-registered in `CAMPAIGN.md` (2026-09-08). This touches every mount/import/DICOM path before the full inference run; it does not deserialize third-party checkpoints or submit.

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
assert torch.cuda.is_available(), 'CUDA unavailable'
assert torch.cuda.device_count() >= 1, 'no CUDA device'
devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('CUDA devices:', devices, flush=True)
ROOT = Path('/kaggle/input')
COMP = ROOT / 'rsna-knee-abnormality-detection'
if not COMP.exists(): COMP = ROOT / 'competitions' / 'rsna-knee-abnormality-detection'
assert (COMP / 'test.csv').is_file(), f'missing test.csv under {COMP}'
assert (COMP / 'test_series.csv').is_file(), f'missing test_series.csv under {COMP}'
series_root = COMP / 'test_series'
if not series_root.is_dir(): series_root = COMP / 'test_images'
assert series_root.is_dir(), f'missing DICOM tree under {COMP}'
mounts = {'raptor-knee-maxspan':'dreaddevelopment','raptor-knee-native384':'dreaddevelopment','raptor-knee-native384dense':'dreaddevelopment','knee-mri-fold-weights':'mattiaangeli','opencv-python-headless-4120088-x86':'mattiaangeli','resnet-50-radimagenet-marwan':'marwanmath','rsna-knee-coat-resgated-ep10-top3':'mattiaangeli','rsna-knee-e11-diverse-heads-v20':'antoinegg1','rsna-knee-e9-radimagenet-heads-v15':'antoinegg1','rsna-knee-llm-labels':'pilkwang','rsna-knee-v52-radimagenet-heads-20260812':'prvsiyan','rsna-knee-weights':'pilkwang'}
def resolve_mount(slug, owner):
    candidates = [ROOT / slug, ROOT / 'datasets' / owner / slug]
    for path in candidates:
        if path.is_dir(): return path
    raise AssertionError(f'missing mount {owner}/{slug}; checked {candidates}')
for slug, owner in mounts.items():
    path = resolve_mount(slug, owner)
    names = [p.name for p in path.rglob('*') if p.is_file()]
    assert names, f'empty mount: {owner}/{slug} at {path}'
    print(f'mount {owner}/{slug}: {path} ({len(names)} files)', flush=True)
test = pd.read_csv(COMP / 'test.csv').head(3).copy()
ids = test['StudyInstanceUID'].astype(str).tolist()
assert len(ids) == 3 and len(set(ids)) == 3
dcm = next(series_root.rglob('*.dcm'), None)
assert dcm is not None, 'no DICOM file found'
header = pydicom.dcmread(dcm, stop_before_pixels=True)
pixel = pydicom.dcmread(dcm).pixel_array
assert pixel.size > 0
targets = [c for c in pd.read_csv(COMP / 'sample_submission.csv', nrows=1).columns if c != 'StudyInstanceUID']
assert len(targets) == 12
submission = pd.DataFrame(0.5, index=range(3), columns=targets)
submission.insert(0, 'StudyInstanceUID', ids)
assert submission.shape == (3, 13) and np.isfinite(submission[targets].to_numpy()).all()
submission.to_csv('/kaggle/working/submission.csv', index=False)
receipt = {'status':'SMOKE_OK','devices':devices,'studies':ids,'dicom':str(dcm),'pixel_shape':list(pixel.shape),'cv2':cv2.__version__,'timm':timm.__version__}
Path('/kaggle/working/smoke_receipt.json').write_text(json.dumps(receipt, indent=2) + '\n')
print('SMOKE_OK', json.dumps(receipt, sort_keys=True), flush=True)
